# 03 · Preprocessing: class filter, features and groups

Having completed some exploration, I'm ready to prepare the data for modelling. That entails three things: removing the classes too sparse to model, building the two feature sets, and defining the groups of related rows that cross-validation has to keep together. That last step is new, and it bears directly on how the results should be read.

Two things that a preprocessing notebook might be expected to do are deliberately absent. The features are not scaled, and the labels are not encoded, because both of those must be fitted on training data only; they happen inside the cross-validation in `04_modeling`. And nothing is written to disk: `04_modeling` repeats these steps with `codon_utils.load_project`, and the last cell of this notebook checks that the two agree.

In [1]:
import numpy as np
import pandas as pd

import codon_utils

In [2]:
# the repaired raw data, with decoded labels (see 01_wrangling)
codon_usage = codon_utils.read_raw('../data/raw/codon_usage.csv')
codon_usage.shape

(13028, 72)

The variables that I'd like to model as outcomes are 'Kingdom' and 'DNAtype'; I'd like to take another look at their value counts.

In [3]:
codon_usage['kingdom'].value_counts()

kingdom
bacteria         2920
virus            2832
plant            2523
vertebrate       2077
invertebrate     1345
mammal            572
bacteriophage     220
rodent            215
primate           180
archaea           126
plasmid            18
Name: count, dtype: int64

In [4]:
codon_usage['dna_type'].value_counts()

dna_type
genomic                   9267
mitochondrial             2899
chloroplast                816
plastid                     31
kinetoplast                  5
cyanelle                     2
unknown                      2
nucleomorph                  2
apicoplast                   2
secondary_endosymbiont       1
chromoplast                  1
Name: count, dtype: int64

Among the DNA types, almost all rows are genomic, mitochondrial or chloroplast; among the kingdoms, every class has at least 126 rows except the plasmids, with 18. As in 2022, I drop the sparse classes, and there are two good reasons beyond their size. First, the headline metric (macro-F1, explained in `04_modeling`) weights every class equally, so a class with between 1 and 31 rows would contribute a per-class score estimated from a handful of test rows, and add mostly noise to the average. Second, 'plasmid' describes a kind of DNA molecule, not a group of organisms, so it doesn't belong among the kingdoms at all.

In [5]:
codon_usage = codon_usage[codon_usage['kingdom'] != 'plasmid']
codon_usage.shape

(13010, 72)

In [6]:
codon_usage = codon_usage[codon_usage['dna_type'].isin(['genomic', 'mitochondrial', 'chloroplast'])]
codon_usage.shape

(12964, 72)

This removed 46 rows: the 44 rows of the rare named DNA types, and the two rows with code 9 ("NA" in the codebook). Every row that leaves is counted; in 2022 the two code-9 rows disappeared later, unrecorded, as missing values.

Cross-tabulating the two labels shows which combinations remain:

In [7]:
codon_usage[['kingdom', 'dna_type']].value_counts()

kingdom        dna_type     
bacteria       genomic          2918
virus          genomic          2832
vertebrate     mitochondrial    1613
plant          genomic          1523
invertebrate   genomic           922
plant          chloroplast       815
mammal         mitochondrial     470
vertebrate     genomic           464
invertebrate   mitochondrial     411
bacteriophage  genomic           220
rodent         mitochondrial     156
plant          mitochondrial     152
archaea        genomic           126
mammal         genomic           102
primate        mitochondrial      97
               genomic            83
rodent         genomic            59
invertebrate   chloroplast         1
Name: count, dtype: int64

One combination stands out as unusual: there's a single observation with 'Kingdom' 'invertebrate' but 'DNAtype' 'chloroplast'. To the best of my knowledge, chloroplasts are exclusive to photosynthetic life, and an invertebrate shouldn't have any. What's going on here?

In [8]:
toxo = (codon_usage['kingdom'] == 'invertebrate') & (codon_usage['dna_type'] == 'chloroplast')
codon_usage.loc[toxo, ['Kingdom', 'DNAtype', 'SpeciesID', 'Ncodons', 'SpeciesName']]

,Kingdom,DNAtype,SpeciesID,Ncodons,SpeciesName
9400,inv,2,5811,6158,chloroplast Toxoplasma gondii


A bit of searching on 'chloroplast Toxoplasma gondii' suggests that what's meant is the apicoplast of *T. gondii*, the parasite responsible for toxoplasmosis: an organelle believed to be the vestige of a chloroplast acquired early in the evolutionary history of its lineage (see https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3166169/). It is not chloroplast DNA in any useful sense, and a single row can't be modelled as a class of its own, so it is dropped, by a condition rather than by a hard-coded row number.

In [9]:
# a boolean mask, not a hard-coded row label
codon_usage = codon_usage[~toxo]
codon_usage.shape

(12963, 72)

In [10]:
codon_usage['kingdom'].value_counts()

kingdom
bacteria         2918
virus            2832
plant            2490
vertebrate       2077
invertebrate     1333
mammal            572
bacteriophage     220
rodent            215
primate           180
archaea           126
Name: count, dtype: int64

In [11]:
codon_usage['dna_type'].value_counts()

dna_type
genomic          9249
mitochondrial    2899
chloroplast       815
Name: count, dtype: int64

That leaves ten kingdoms and three DNA types, and 12,963 rows. One more check, for missing values:

In [12]:
codon_usage.isna().sum().sum()

np.int64(0)

None; the two code-9 rows that appeared as missing values in 2022 (see `01_wrangling`) were removed, and counted, in the DNA-type step above.

`codon_utils.filter_classes` applies the same three steps, and records the row count after each, so that the modelling notebook can report exactly how the 13,028 rows became 12,963:

In [13]:
model_rows, steps = codon_utils.filter_classes(codon_utils.read_raw('../data/raw/codon_usage.csv'))
assert model_rows.index.equals(codon_usage.index)  # same rows as the steps above
steps

,step,rows,removed
0,raw,13028,0
1,drop Kingdom 'plm' (plasmid),13010,18
2,"keep DNAtype in {0 genomic, 1 mitochondrial, 2...",12964,46
3,drop inv + chloroplast (Toxoplasma apicoplast)...,12963,1


## Scaling happens inside cross-validation

In 2022 I moved the scaler after the train/test split to avoid leakage; here it moves inside cross-validation. k-NN and the MLP depend on the scale of the features (through distances and gradient steps respectively), while tree models don't care. A scaler fitted to all the rows would let the test rows help shape the transformation applied to the training rows, a small leak but a real one. So in `04_modeling`, k-NN and the MLP are pipelines whose `StandardScaler` is fitted on the training folds only, separately for each fold.

## Features

These are the two feature sets compared in `04_modeling`: the 64 codon frequencies, and the 21 amino-acid groups built in `01_wrangling` (which, as `02_eda` showed, are strictly codon groups under the standard genetic code).

In [14]:
amino = codon_utils.amino_features(codon_usage)
print('codon features:', len(codon_utils.CODONS), ' amino-acid features:', amino.shape[1])

codon features: 64  amino-acid features: 21


In [15]:
amino.mean()

alanine          0.074558
arginine         0.046394
asparagine       0.044484
aspartic_acid    0.045354
cysteine         0.014892
glutamine        0.034860
glutamic_acid    0.049968
glycine          0.065275
histidine        0.023725
isoleucine       0.071649
leucine          0.108341
lysine           0.049952
methionine       0.021139
phenylalanine    0.048169
proline          0.049957
serine           0.071234
threonine        0.062768
tryptophan       0.011609
tyrosine         0.034326
valine           0.062914
stop             0.008432
dtype: float64

In [16]:
amino.std()

alanine          0.024743
arginine         0.019463
asparagine       0.014037
aspartic_acid    0.017570
cysteine         0.010464
glutamine        0.014233
glutamic_acid    0.019886
glycine          0.017715
histidine        0.007687
isoleucine       0.032335
leucine          0.036655
lysine           0.021792
methionine       0.008158
phenylalanine    0.017158
proline          0.014652
serine           0.015627
threonine        0.019276
tryptophan       0.006574
tyrosine         0.010434
valine           0.015594
stop             0.011055
dtype: float64

These are raw frequencies, not standard scores: the means range from 0.008 (stop) to 0.108 (leucine), and the standard deviations from 0.007 to 0.037. (The 2022 notebook described standardized output at this point, although the scaler had been commented out a few cells earlier.)

## Labels

The labels stay as strings here. The cross-validation code in `04_modeling` encodes them as integers, in the same way for every model. In 2022 I fitted k-NN and random forest on one-hot target columns (`pd.get_dummies`). scikit-learn treats a one-hot target as ten separate yes/no problems, one per class, so a row could receive no label at all; those rows counted as errors, and the two models were scored on a subtly different task from the others (see `ERRATUM.md`).

## Groups for cross-validation

If closely related rows are split between the training and test folds, a model can score well simply by recognising near-copies of what it has already seen, and the test score then measures memory rather than generalisation to new organisms. To prevent that, cross-validation needs to know which rows belong together. I use two levels of grouping:

- *species*: rows with the same `SpeciesID` (which is the NCBI taxonomy id) form one group;
- *genus*: approximated by the first informative word of the species name (`codon_utils.name_token`), and made constant within each `SpeciesID`, so that a genus group always contains whole species.

In [17]:
groups = codon_utils.add_groups(codon_usage)
print('species groups:', groups['group_species'].nunique(), ' genus groups:', groups['group_genus'].nunique())
groups['group_genus'].value_counts().head()

species groups: 12331  genus groups: 4548


group_genus
human             388
streptomyces      186
tomato            154
pseudomonas       150
enterobacteria    146
Name: count, dtype: int64

There are 12,331 species groups and 4,548 genus groups. The genus heuristic is serviceable for cellular organisms but rough for viruses: the largest group, 'human', collects every virus whose name begins with "Human", from herpesviruses to papillomaviruses. Since `SpeciesID` is the NCBI taxonomy id, proper genus, family and order groups could be built from the NCBI taxonomy itself; that is the obvious next refinement (see the README).

But first: what do the rows that share a `SpeciesID` actually have in common?

In [18]:
multi = groups[groups['SpeciesID'].duplicated(keep=False)]
print(multi['SpeciesID'].nunique(), 'SpeciesIDs have more than one row;',
      'rows that repeat a (SpeciesID, DNA type) pair:', groups.duplicated(['SpeciesID', 'DNAtype']).sum())
multi.groupby('SpeciesID')['dna_type'].agg(lambda s: ' + '.join(sorted(s))).value_counts()

596 SpeciesIDs have more than one row; rows that repeat a (SpeciesID, DNA type) pair: 0


dna_type
genomic + mitochondrial                  423
chloroplast + genomic                    129
chloroplast + genomic + mitochondrial     36
chloroplast + mitochondrial                8
Name: count, dtype: int64

They are always different compartments of one organism: 423 organisms with a nuclear and a mitochondrial row, 129 with nuclear and chloroplast, 36 with all three. No `SpeciesID` repeats a DNA type. So is a shared `SpeciesID` a source of leakage? That depends on whether the codon profiles of one organism's compartments resemble each other. Let's compare the L1 distance between two rows with the same `SpeciesID` with the distance between random pairs of rows:

In [19]:
X = groups[codon_utils.CODONS].to_numpy()
position = pd.Series(np.arange(len(groups)), index=groups.index)
same = [np.abs(X[position[g.index[0]]] - X[position[g.index[1]]]).sum()
        for _, g in multi.groupby('SpeciesID')]
rng = np.random.default_rng(0)
a, b = rng.integers(0, len(X), 20_000), rng.integers(0, len(X), 20_000)
random_pairs = np.abs(X[a[a != b]] - X[b[a != b]]).sum(axis=1)
print(f'median L1 distance: same SpeciesID {np.median(same):.3f} ({len(same)} pairs), '
      f'random rows {np.median(random_pairs):.3f} ({len(random_pairs):,} pairs)')

median L1 distance: same SpeciesID 0.771 (596 pairs), random rows 0.699 (19,999 pairs)


They don't: two rows from one organism are, if anything, slightly *further* apart than two random rows (median L1 distance 0.77 against 0.70). A nuclear and a mitochondrial codon profile differ about as much as those of two unrelated organisms do, which is not so surprising given the previous notebook: mitochondria even use a different genetic code. So a `SpeciesID` shared between training and test folds should leak very little, and `04_modeling` measures it.

Relatedness between *different* `SpeciesID`s is another matter. Strains and close relatives are often filed under separate ids, and some of them have identical codon vectors:

In [20]:
key = groups[codon_utils.CODONS].round(5).astype(str).agg('|'.join, axis=1)
twins = groups[key.duplicated(keep=False)].assign(vector=key)
by_vector = twins.groupby('vector')
print(len(twins), 'rows form', by_vector.ngroups, 'groups of identical codon vectors;',
      (by_vector['SpeciesID'].nunique() > 1).sum(), 'span several SpeciesIDs and',
      (by_vector['group_genus'].nunique() == 1).sum(), 'lie within one genus group')
by_vector['SpeciesName'].agg(' | '.join).head(6).to_frame().reset_index(drop=True)

63 rows form 23 groups of identical codon vectors; 23 span several SpeciesIDs and 20 lie within one genus group


,SpeciesName
0,Bordetella hinzii BC-306 | Bordetella hinzii L...
1,Sphingomonas sp. GTIN11 | Klebsiella sp. LSSE-H2
2,Measles virus strain Moraten | Measles virus s...
3,Salmonella phage SETP7 | Salmonella phage SETP11
4,Cyanophage Ma-LMM02 | Cyanophage Ma-LMM03 | Cy...
5,Norovirus Hu/NLV/Oxford/B4S2/2002/UK | Norovir...


Here are near-copies under different ids: two *Bordetella hinzii* strains, the Moraten and Schwarz measles vaccine strains, a series of Salmonella phages and cyanophages. A split that holds out species would put such pairs on both sides of the line, while a split that holds out genus groups mostly would not (20 of these 23 sets of identical vectors lie within one genus group). That's why `04_modeling` evaluates three fold schemes: random folds, species held out, and genus held out. The difference between the last two measures how much of a model's apparent skill comes from having seen a close relative.

Finally, `04_modeling` rebuilds this frame with `codon_utils.load_project` (`read_raw`, then `filter_classes`, then `add_groups`). Let's check that it gives the same rows and groups:

In [21]:
project, _ = codon_utils.load_project('../data/raw/codon_usage.csv')
assert project.index.equals(groups.index)
assert (project['group_genus'] == groups['group_genus']).all()
print('load_project matches this notebook:', len(project), 'rows')

load_project matches this notebook: 12963 rows
